# Fit Multiple Files Independently

Fit six datasets with the same model, each on its own — no parameters tied across files. This is the bridge between single-file fitting (`01_basic_fitting`) and shared multi-file fitting (`21_multi_file_shared_fit`): the setup is shared, the fits are not.

Running this through a `Project` instead of a hand-rolled `for f in files:` loop buys four things: setup applied once, one coherent export tree, one-call cross-file comparison, and one portable HDF5 archive. Each shows up in the workflow below.

**Workflow:**

1. Load six files into one Project
2. Shared setup across all files (fit limits, baseline window, model)
3. Per-file baseline fits
4. Per-file 2D fits
5. Cross-file comparison
6. Export and archive the whole batch

The data is the kicked-decay dataset from `21_multi_file_shared_fit` (6 files, `expFun A` from 5 down to 0.1), loaded here by relative path to avoid duplication.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import trspecfit

## 1. Load Six Files Into One Project

In [ ]:
project = trspecfit.Project(path=Path.cwd(), name="per_file_batch")

# Data lives next door so we don't duplicate the 6 CSVs in this example dir.
data_root = project.path.parent / "21_multi_file_shared_fit" / "data"

energy = np.loadtxt(data_root / "energy.csv")
time = np.loadtxt(data_root / "time.csv")

shift_amplitudes = [5, 2, 1, 0.5, 0.2, 0.1]
files = []
for i, amp in enumerate(shift_amplitudes, start=1):
    file_data = np.loadtxt(data_root / f"data_{i}.csv", delimiter=",")
    f = trspecfit.File(
        parent_project=project,
        path=f"data_{i}",
        data=file_data,
        energy=energy,
        time=time,
    )
    files.append(f)
    print(f"File {i}: expFun A = {amp}, shape {file_data.shape}")

# detail=1 also lists the attached files and plots each 2D data grid
project.describe(detail=1)

## 2. Shared Setup Across All Files

Same fit limits, same baseline window, same model loaded onto every file — one call each. This is the per-file boilerplate that a bare `for f in files: ...` loop would have to repeat.

In [ ]:
project.set_fit_limits(energy_limits=[5, 18], time_limits=[-20, 99])
# time_type="ind": indices, not times. Slices 0-22 are t = -20 ... -9, which ends
# three IRF widths (SD = 3) before t = 0 so the onset cannot bleed into the baseline
project.define_baselines(time_start=0, time_stop=22, time_type="ind")
project.load_models(model_yaml="models_energy.yaml", model_info="base")

## 3. Per-File Baseline Fits

`project.fit_baselines` is a thin loop over `file.fit_baseline`, so each baseline is fit independently.

In [ ]:
project.fit_baselines(
    model_name="base",
    stages=2,  # simplex pass, then leastsq refinement
    try_ci=0,  # skip confidence intervals: see 12_uncertainty_mcmc
)

## 4. Per-File 2D Fits

For the 2D step we deliberately use a per-file loop (`for f in files: f.fit_2d(...)`) instead of `project.fit_2d(...)`. The latter is the **shared-parameter** path (notebook `21_multi_file_shared_fit`); here we want every file's `tau`, `SD`, `A` fit independently.

The 2D YAML uses `vary: project/file/static` strings (carried over from notebook 21). For per-file fits those are reduced to a simple True/False vary level (`project` and `file` both → True; `static` → False), so the same YAML works in either workflow.

In [ ]:
project.load_models(model_yaml="models_energy.yaml", model_info="2D")
project.add_time_dependences(
    target_model="2D",
    target_parameter="GLP_01_x0",
    dynamics_yaml="models_time.yaml",
    dynamics_model="MonoExpPosIRF",
)

for f in files:
    f.fit_2d(
        model_name="2D",
        stages=2,  # simplex pass, then leastsq refinement (fit_2d alone defaults to 1)
        try_ci=0,  # skip confidence intervals: see 12_uncertainty_mcmc
    )

## 5. Cross-File Comparison

Omit `file=` to survey the whole batch: `project.results.compare_models(fit_type="2d")` reads the in-session fit history (no disk I/O) and returns one row per `(file, model, fit_type)` slot. The `file=` filter takes a single file name or `File` to zoom into one row; loop over it to inspect a subset.

With one model and one fit type per file, the comparison becomes a **per-file fit-quality survey**: which files fit best and which carry the largest residuals. Parameter values are not in this table; `project.results.get_parameters(file=..., fit_type=...)` returns them per file, as the next cell does. For a true model-vs-model comparison see [`10_model_comparison`](../10_model_comparison/example.ipynb).

In [ ]:
project.results.compare_models(fit_type="2d")

The survey ranks fit quality but shows no parameters. Each file was generated with a known kick amplitude (the `shift_amplitudes` printed in section 1) and the shared truth `tau = 50`, `SD = 3`, so a short loop over `get_parameters` pairs what every independent fit recovered with what went in:

In [ ]:
# what each independent 2D fit recovered, next to the truth it was generated with
rows = []
for f, amp in zip(files, shift_amplitudes):
    fit_df = project.results.get_parameters(file=f.name, fit_type="2d")
    fit_df = fit_df.set_index("name")
    rows.append({
        "file": f.name,
        "A truth": amp,
        "A fit": fit_df.loc["GLP_01_x0_expFun_01_A", "value"],
        "A stderr": fit_df.loc["GLP_01_x0_expFun_01_A", "stderr"],
        "tau fit (truth 50)": fit_df.loc["GLP_01_x0_expFun_01_tau", "value"],
        "tau stderr": fit_df.loc["GLP_01_x0_expFun_01_tau", "stderr"],
        "SD fit (truth 3)": fit_df.loc["GLP_01_x0_gaussCONV_SD", "value"],
        "SD stderr": fit_df.loc["GLP_01_x0_gaussCONV_SD", "stderr"],
    })
pd.DataFrame(rows).round(3)

The four strong files recover `A` to about a percent and `tau` and `SD` to a few percent; the two weakest miss by 10 to 20 %, and their `stderr` say so: the `stderr` on `tau` grows fiftyfold from the strongest file to the weakest, because a 0.1 kick barely constrains a time constant or an IRF width on its own. When those two are the same physical quantity in every file, sharing them is the fix, and that is what [`21_multi_file_shared_fit`](../21_multi_file_shared_fit/example.ipynb) does.

## 6. Export and Archive the Whole Batch

Two methods, two audiences:

- **`project.export_fits()` → CSV + PNG tree.** One coherent directory rooted at `<path>/<file_name>/<model>__<fit_type>/...` — easier to diff or zip than N per-file dumps. One-way; no `load` counterpart.
- **`project.save_fits()` → single HDF5 archive.** Lossless, σ-snapshot included, round-trips back into trspecfit via `FitResults.load`.

Both calls take `file=` / `model=` / `fit_type=` filters and `select=` to ship a subset instead of the whole batch. See [`11_save_load_export`](../11_save_load_export/example.ipynb) for the filtered-export and round-trip details.

In [ ]:
project.export_fits("batch_export", overwrite=True)
project.save_fits("batch.fit.h5", overwrite=True)

loaded = trspecfit.FitResults.load("batch.fit.h5")
print(loaded)
print("\nfiles:  ", loaded.files())
print("models: ", loaded.models())

## Tips

- **Choose the right `Project` method.** `project.fit_baselines()` and `project.load_models()` are setup helpers — loops that share boilerplate. `project.fit_2d()` is a true shared-parameter fit (see `21_multi_file_shared_fit`). Use a bare `for f in files: f.fit_2d(...)` loop for independent 2D fits, as shown here.
- **Survey vs zoom.** `compare_models()` with no `file=` returns one row per slot across the whole project — the cross-file survey. `compare_models(file=name)` filters to one file (single-target only; loop to inspect a subset).

## Next Steps

- Share parameters across files (instrument response, decay constants): [`21_multi_file_shared_fit`](../21_multi_file_shared_fit/example.ipynb). The YAMLs here already carry the `project/file/static` flags that drive `project.fit_2d()`.
- Filtered export, `select=`, and the archive round trip: [`11_save_load_export`](../11_save_load_export/example.ipynb)
- Model-vs-model comparison on one file: [`10_model_comparison`](../10_model_comparison/example.ipynb)